In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

In [ ]:
try:
    spark.stop()
except:
    pass

In [ ]:
spark = (SparkSession.builder.appName("HealthcareDataProcessing_DimDate")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.memory.offHeap.enabled", "true")
    .config("spark.memory.offHeap.size", "2g")
.getOrCreate())

In [ ]:
# for this project I am creating this notebook to create dimdate dimension table for 70 years, 1970 to 2040. 

from datetime import date, datetime


start_date = datetime(1920,1,1,0,0,0).timestamp()
end_date = datetime(2040,12,31,0,0,0).timestamp()

silver_dim_date_path = "../../data_lake/silver/dim_date"

In [ ]:
dim_date = (spark.range(
    int(start_date), 
    int(end_date), 
    86400, 
).select(
    F.from_unixtime("id").cast("date").alias("date")
).withColumn("date_key", F.date_format("date", "yyyyMMdd").cast("int")) 
    .withColumn("year", F.year("date")) 
    .withColumn("month", F.month("date")) 
    .withColumn("day", F.dayofmonth("date"))
    .withColumn("quarter", F.quarter("date"))
    .withColumn("month_name", F.date_format("date", "MMMM")) 
    .withColumn("day_of_week", F.date_format("date", "E")) 
    .withColumn("day_of_year", F.dayofyear("date")) 
    .withColumn("week_of_year", F.weekofyear("date")) 
    .withColumn("is_weekend", F.when(F.date_format("date", "E").isin("Sat", "Sun"), True).otherwise(False))
).withColumn("silver_timestamp", F.current_timestamp())


In [ ]:
dim_date.write.format("parquet").mode("overwrite").save(silver_dim_date_path)

In [ ]:
spark.stop()